# **1. 이안류 CCTV 데이터셋**
​AI Hub의 '이안류 CCTV 데이터'는 우리나라 주요 해수욕장(해운대, 송정, 대천, 중문, 낙산)에서 이안류 발생 여부와 위치를 모니터링하기 위해 구축된 인공지능 학습용 데이터셋입니다. 해수욕장 주변에 설치된 CCTV 영상을 이미지로 변환하여, 이안류 발생 여부와 위치를 가시화하는 모델 개발에 활용할 수 있습니다. 이 데이터셋은 이안류 탐지 및 예측 시스템 개발에 필수적인 자료를 제공하며, 해수욕객의 안전을 위한 응용 서비스 구성에 활용될 수 있습니다. ​이안류는 해안에서 먼 바다로 빠르게 이동하는 폭이 좁은 바닷물의 흐름으로, 기상 상태가 양호한 경우에도 나타나며, 얕은 곳에 있던 해수욕객을 순식간에 수심이 깊은 먼 바다로 이동시켜 인명사고를 유발할 수 있습니다. 따라서 이러한 데이터셋은 해수욕장 안전 관리 및 이안류 예측 모델 개발에 중요한 역할을 합니다.

아래 AIHub에서 이안류 CCTV 데이터셋을 다운로드 받습니다.

https://www.aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=data&dataSetSn=71297

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import json  # 라벨이 json이라 
import shutil # 파일 관련 모듈(삭제, 복사 등)
import yaml
from ultralytics import YOLO
from pathlib import Path


# 2. YOLO 데이터셋 만들기
### 1. JSON Bounding Box > YOLO Bounding Box
- 원본 JSON의 drawing에는 객체를 둘러싼 사각형의 점 좌표가 들어있음
- YOLO의 일반적인 객체 탐지 라벨 형식
    - 클래스 ID X센터 Y센터 너비 높이
    - 네 개의 좌표를 모두 0 ~ 1사이로 정규화

In [ ]:
BASE_ROOT = "/content/drive/MyDrive/Colab Notebooks/dataset"

In [ ]:
BASE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/dataset")
# VOC_ROOT = DATA_DIR / "VOC"
YOLO_ROOT = DATA_DIR / 'Custom'

# ANNOTATIONS_DIR = VOC_ROOT / "Annotations"
# IMAGES_DIR = VOC_ROOT / "JPGImages"

In [ ]:
for sub in [
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val", "labels/test"
]:
    (YOLO_ROOT/sub).mkdir(parents=True, exist_ok=True)

In [ ]:
# Annotations에서 Object가 No, Yes 그리고 class가 0, 1로 되어 있음
VOC_CLASSES = [
    "Yes"
]

# 예측 클래스 명칭(문자열)을 고유한 번호표(ID) 값으로 매핑
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
CLASS_TO_ID


In [ ]:
# 이미지 좌상단/우하단 좌표를 YOLO 중심점 데이터로 연산
def voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h):
    x_center = ((xmin + xmax)/2.0) / image_w
    y_center = ((ymin + ymax)/2.0) / image_h
    box_w = (xmax - xmin) / image_w
    box_h = (ymax - ymin) / image_h
    return x_center, y_center, box_w, box_h

In [ ]:
def convert_voc_json(json_path, label_path):   
    root = json.loads(Path(json_path).read_text(encoding="utf-8"))
    
    if root["annotations"]["object"] != "Yes":
        return
    
    resolution = root["image_info"]["resolution"]
    image_w, image_h = map(float, resolution.split(","))

    drawing = root["annotations"]["drawing"]
    
    lines = []

    for box in drawing:
        xs = [point[0] for point in box]
        ys = [point[1] for point in box]
        
        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        xmin = max(0.0, min(xmin, image_w))
        xmax = max(0.0, min(xmax, image_w))
        ymin = max(0.0, min(ymin, image_h))
        ymax = max(0.0, min(ymax, image_h))

        if xmax <= xmin or ymax <= ymin:
            continue
        
        x, y, w, h = voc_box_to_yolo(xmin, ymin, xmax, ymax, image_w, image_h)
        class_id = CLASS_TO_ID["Yes"]
        lines.append(f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines), encoding="utf-8")

In [ ]:
def prepare_split(split_name):
    split_root = DATA_DIR / split_name
    annotation_root = split_root / "라벨링데이터"
    image_root = split_root / "원천데이터"
    
    out_image_dir = YOLO_ROOT / "images" / split_name
    out_label_dir = YOLO_ROOT / "labels" / split_name

    out_image_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok = True)

    json_files = {p.stem: p for p in annotation_root.rglob("*.json")}
    image_files = {p.stem: p for p in image_root.rglob("*.jpg")}
    image_ids = list(json_files)

    # print(image_ids)

    for image_id in image_ids:
        src_image = image_files[image_id]
        src_json = json_files[image_id]
        
        shutil.copy2(src_image, out_image_dir / src_image.name)
        convert_voc_json(src_json, out_label_dir / f"{image_id}.txt")
    print(f'{split_name}: {len(image_ids):,} images prepared')

In [ ]:
prepare_split("train")
prepare_split("val")

### 2. 원본 이미지와 JSON 라벨을 연결한 뒤 train/val/test 폴더로 나눔
- 같은 원본 영상에서 나온 프레임이 train과 test에 동시에 들어가면 성능이 실제보다 높게 보일 수 있음
- 파일명 앞부분을 그룹으로 사용해 가능한 하나의 같은 영상 계열이 한 세트에만 들어가도록 분리
- 데이터 수가 적으면 YOLO 성능이 낮아질 수 있음
    - 전체 이미지: 360wkd
    - 이안류 객체가 있는 이미지: 240장
    - 배경 이미지: 120장

> YOLO는 단순히 이미지를 외우는 것이 아니라, 려러 이미지에서 반복적으로 나타나는 시각적 특징을 학습하여 새로운 이미지에서도 객체를 찾아야 함. 단순한 파일 개수가 아니라 서로 다른 상황을 얼마나 다양하게 포함하고 있는지 중요

- cctv 연속 프레임에서는 무작위로 이미지를 섞어 나누는 방법을 사용하지 않음
    - 서로 몇 초 차이인 것의 동일한 프레임이 Train과 Test에 동시에 들어가면 모델은 Test 이미지를 처음 보는 것이 아니라, 학습 때 본 장면과 매우 유사한 정면을 다시 보는 것과 비슷해짐(데이터 누수)

- 전체 데이터셋을 확보할 수 있다면 아래 단위로 분리하는 것이 좋음
    - CCTV 카메라 각도
    - 해수욕장 위치
    - 날짜
    - 촬영 시간대

In [ ]:
custom_voc_yaml = YOLO_ROOT / "custom_voc.yaml"

data_config = {
    "path": str(YOLO_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    # "test": "images/test",
    "names": {i: name for i, name in enumerate(VOC_CLASSES)}
}

with open(custom_voc_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False) # 알파벳 순으로 정렬할 필요가 없어서 False

print("생성된 YAML 경로: ", custom_voc_yaml)
print(custom_voc_yaml.read_text(encoding='utf-8'))

In [ ]:
model = YOLO("yolo11s.pt") # pt는 모델 아키텍처까지 있고 pth는 가중치만 있음
model

In [ ]:
results = model.train(
    data = str(custom_voc_yaml),
    epochs = 10,
    imgsz = 640,
    batch = 16,
    # device = DEVICE,
    workers = 8,
    project = "/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name = "이안_yolo11s",
    exist_ok = True,
)

In [ ]:
best_path = Path("/content/drive/MyDrive/Colab Notebooks/train_result/이안류/이안_yolo11s/weights/best.pt")
model = YOLO(str(best_path))

In [ ]:
val_results = model.val(
    data = str(custom_voc_yaml),
    split = 'val',
    imgsz = 640,
    batch = 16,
    # device = DEVICE,
    workers = 8
)

In [ ]:
print('mAP50: ', val_results.box.map50)
print('mAP50-95: ', val_results.box.map)
print('mAP75: ', val_results.box.map75)

In [ ]:
import random

val_images = list((YOLO_ROOT / "images" / "val").glob("*.jpg"))
sample_images = random.Random(2026).sample(val_images, 5)

pred_results = model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.30,
    save=True,
    project="/content/drive/MyDrive/Colab Notebooks/train_result/이안류",
    name="test_images_predict",
    exist_ok=True
)

print(f"예측 이미지 수: {len(pred_results)}")